**DOMÍNIO DE INFLUÊNCIAS** Latitude: 30°N  → 30°S
Longitude: 100°W → 20°E

Atlântico tropical norte, Atlântico tropical sul, costa do Nordeste brasileiro, Golfo da Guiné, Caribe tropical

**ENEB** Latitude: 2°S → 12°S
Longitude: 34°W → 40°W

**GRANDE_RECIFE** Latitude: 7.50°S → 8.50°S
Longitude: 35.50°W → 34.25°W

# PLANO DE AÇÃO COLAB: DESCOBERTA DE INFLUÊNCIAS CLIMÁTICAS NO ENEB (Arquitetura ETL & Zarr)

## ETAPA 1: Preparação do Ambiente e APIs
1. **Instalação das Bibliotecas:** `!pip install xarray dask netCDF4 cdsapi eofs shap scikit-learn xgboost matplotlib cartopy zarr`
2. **Montagem do Disco:** Conectar o Google Drive (`drive.mount('/content/drive')`) para garantir que os dados processados não sejam perdidos ao fechar o Colab.
3. **Autenticação:** * Configurar o arquivo `~/.cdsapirc` com a chave da API do Copernicus Climate Data Store (CDS) para o ERA5.
   * Configurar credenciais do Copernicus Marine Service (CMEMS) para o ORAS5.

## ETAPA 2: Aquisição Dinâmica (Extração Estratégica)
*A extração ocorrerá através de um laço de repetição (loop) anual, evitando o esgotamento do disco virtual.*

1. **Atmosfera - ERA5 (via `cdsapi` - datasets: 'reanalysis-era5-single-levels' e 'pressure-levels'):**
   * *Configuração de Tempo:* Baixar os 4 horários sinóticos principais (00:00, 06:00, 12:00, 18:00) para compor o ciclo diário.
   * *Download Isolado de Fluxo (Alvo):* Precipitação Total (tp) para o domínio alvo (ENEB: 2°S–12°S, 34°W–40°W) em arquivo `.nc` temporário separado, garantindo o cálculo de soma diária (acumulada) na transformação.
   * *Download Agrupado de Superfície (Single Levels):* Pressão ao Nível do Mar (msl), Ventos a 10m (u10, v10), Temperatura a 2m (t2m), CAPE, Fluxos de Calor Latente e Sensível, e Água Precipitável Total (TCWV).
   * *Download Agrupado de Altitude (Pressure Levels - 850, 500 e 200 hPa):* Ventos Zonal e Meridional (u, v), Umidade Específica (q), Geopotencial (z) e Velocidade Vertical (w).
   * *Domínio Preditivo (Superfície e Altitude):* Atlântico Tropical (30°N–30°S, 100°W–20°E).

2. **Oceano - ORAS5 (via `cdsapi` ou FTP do CMEMS - dataset: 'ocean-reanalysis-oras5'):**
   * *Variáveis Termodinâmicas:* Temperatura da Superfície do Mar (SST), Profundidade da Camada Misturada (MLD), Conteúdo de Calor Oceânico (OHC) e Profundidade da Isoterma de 20°C (Z20).
   * *Variáveis Dinâmicas e Físicas:* Correntes Oceânicas Superficiais (u, v), Salinidade Superficial e Altura da Superfície do Mar (SSH).
   * *Domínio Preditivo:* Atlântico Tropical (30°N–30°S, 100°W–20°E).
   * *Frequência:* Utilizar o produto consolidado mensal, aproveitando a alta inércia térmica do oceano e poupando processamento e espaço em disco.

## ETAPA 3: Transformação e Carregamento (Pipeline ETL & Zarr)
1. **Resampling Diário Imediato no Xarray:**
   * *Variáveis de Estado:* Aplicar `ds.resample(time='1D').mean()` para obter a média diária real da pressão, umidade e ventos.
   * *Precipitação:* Aplicar rigorosamente `ds.resample(time='1D').sum()` para calcular o volume diário acumulado.
2. **Fusão e Otimização Zarr:**
   * Mesclar os datasets transformados (`xr.merge()`).
   * Aplicar fatiamento espacial (*chunking*, ex: `latitude: 50, longitude: 50`) para otimizar a leitura paralela.
   * Salvar diretamente no Google Drive usando `to_zarr()`.
3. **Limpeza de Disco (Drop):** Executar `os.remove()` nos arquivos `.nc` brutos imediatamente após a conversão para `.zarr`.
4. **Alinhamento Espacial:** Executar *regridding* com `xarray.interp()` para alinhar as grades do ORAS5 e ERA5 (ex: 0.25° x 0.25°).
5. **Cálculo de Anomalias e Normalização:** Subtrair a climatologia base (`groupby('time.month')`) para expor os desvios extremos e aplicar Z-score para equalizar grandezas físicas distintas.

## ETAPA 4: Machine Learning Espacial (Mapeando a Origem)
*O objetivo é identificar geograficamente os clusters oceânicos que controlam a chuva no ENEB.*
1. **Análise de Componentes Principais (PCA / EOFs Espaciais):**
   * Aplicar `eofs.xarray` nas anomalias de TSM e Pressão para isolar matematicamente as assinaturas do Atlântico Tropical, sem depender de índices estáticos.
2. **Correlação de Ponto a Campo com Defasagem (Lagged Correlation):**
   * Cruzar a série temporal da precipitação do ENEB com as matrizes 3D de TSM e Ventos, aplicando defasagens (lags) temporais (ex: 1, 2, 3 meses).
3. **Pixel-wise Random Forest:**
   * Achatar espacialmente a grade do Atlântico para um formato tabular.
   * Treinar o modelo para prever a chuva no ENEB e reconstruir a *Feature Importance* em formato 2D, gerando um mapa das coordenadas oceânicas críticas.

## ETAPA 5: Machine Learning Explicativo (Ranqueando a Física)
*O objetivo é definir qual grandeza termodinâmica ou dinâmica tem o maior peso na geração de chuvas.*
1. **Criação do Dataset Tabular Final:**
   * *Alvo:* Chuva diária acumulada no ENEB.
   * *Features:* Séries temporais extraídas das áreas críticas identificadas na Etapa 4 (ex: Índice de TSM regional, intensidade local dos Alísios, anomalia da ASAS).
2. **Treinamento com Gradient Boosting:**
   * Utilizar XGBoost ou LightGBM para capturar as interações atmosféricas não-lineares.
3. **Desmontagem Interpretativa (SHAP):**
   * Extrair os valores via `shap.TreeExplainer()` e plotar o *SHAP Summary Plot*.
   * Consolidar o ranking final: identificação do peso exato de cada variável e a direcionalidade de seu impacto (se a anomalia atua favorecendo ou inibindo a precipitação).

In [1]:
pip install xarray[complete] dask netCDF4 cdsapi zarr pandas

  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/801.4 kB ? eta -:--:--
   ---------------------------------------- 801.4/801.4 kB 4.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 16.0 MB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   ----- ---------------------------------- 5.2/38.1 MB 26.5 MB/s eta 0:00:02
   ------------ --------------------------- 12.3/38.1 MB 29.7 MB/s eta 0:00:01
   ----------------- ---------------------- 16.8/38.1 MB 32.0 MB/s eta 0:00:01
   --------------------------- ------------ 26.0/38.1 MB 31.6 MB/s eta 0:00:01
   ----------------------------------- ---- 33.6/38.1 MB 32.3 MB/s eta 0:00:01
   ---------------------------------------  38.0/38.1 MB 32.7 MB/s eta 0:00:01
   ---------------------------------------- 38.1/38.1 MB 29.6 MB/s  0:00:01
   -----------

In [ ]:
import os
import cdsapi
import xarray as xr
import pandas as pd
import time
import zipfile

# 1. CONFIGURAÇÕES E CLIENTE
# ------------------------------------------------------------------------------
CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = "e070e6f5-c003-4a24-8d53-c40d433402cd"
c = cdsapi.Client(url=CDS_URL, key=CDS_KEY)

BASE_DIR = os.getcwd()
CACHE_DIR = os.path.join(BASE_DIR, "cache_nc")
DATA_DIR = os.path.join(BASE_DIR, "data_zarr")
for d in [CACHE_DIR, DATA_DIR]: os.makedirs(d, exist_ok=True)

ANO = '1980'
AREA = [30, -100, -30, 20] # N, W, S, E

# 2. DICIONÁRIOS COM NOMES EXTENSOS (TOTAL 20 VARIÁVEIS)
# ------------------------------------------------------------------------------
VARS_ERA5_SL = {
    'total_precipitation': 'Total-Precipitation',
    '2m_temperature': 'Temperature-At-2-Meters',
    'sea_surface_temperature': 'Sea-Surface-Temperature-ERA5', 
    'mean_sea_level_pressure': 'Mean-Sea-Level-Pressure',
    '10m_u_component_of_wind': 'Zonal-Wind-At-10-Meters',
    '10m_v_component_of_wind': 'Meridional-Wind-At-10-Meters',
    'convective_available_potential_energy': 'Convective-Available-Potential-Energy',
    'surface_latent_heat_flux': 'Surface-Latent-Heat-Flux',
    'surface_sensible_heat_flux': 'Surface-Sensible-Heat-Flux',
    'total_column_water_vapour': 'Total-Column-Water-Vapour'
}

VARS_ERA5_PL = {
    'u_component_of_wind': 'Zonal-Wind-Altitude',
    'v_component_of_wind': 'Meridional-Wind-Altitude',
    'specific_humidity': 'Specific-Humidity-Altitude',
    'geopotential': 'Geopotential-Height',
    'vertical_velocity': 'Vertical-Velocity-Omega'
}

VARS_ORAS5 = {
    'sea_surface_temperature': 'Sea-Surface-Temperature-ORAS5',
    'mixed_layer_depth_0_01': 'Mixed-Layer-Depth-ORAS5',
    'ocean_heat_content_for_the_upper_300m': 'Ocean-Heat-Content-300-Meters-ORAS5',
    'salinity': 'Sea-Water-Salinity-ORAS5', 
    'sea_surface_height': 'Sea-Surface-Height-ORAS5'
}

# 3. MOTOR DE DOWNLOAD (PADRÃO OFICIAL CDSAPI .download())
# ------------------------------------------------------------------------------
def download_engine(dataset, var_key, source, full_name):
    target_nc = os.path.join(CACHE_DIR, f"{source}_{full_name}_{ANO}.nc")
    target_zip = target_nc.replace(".nc", ".zip")
    
    if os.path.exists(target_nc):
        try:
            # Tenta abrir. Se conseguir, o arquivo está ok.
            with xr.open_dataset(target_nc, engine='netcdf4') as tmp:
                return target_nc
        except Exception:
            # Se der erro (arquivo corrompido), fecha tudo e tenta deletar
            print(f"⚠️ Arquivo corrompido detectado: {full_name}. Tentando baixar novamente...")
            # O truque aqui é não tentar o os.remove dentro de um bloco que ainda pode segurar o arquivo
            pass 
        
        # Tenta deletar fora do bloco try anterior para garantir liberação
        try:
            os.remove(target_nc)
        except:
            print(f"❌ Não foi possível deletar {target_nc}. Por favor, apague-o manualmente.")
            return None
        
    request = {'variable': [var_key], 'year': [ANO], 'month': [f"{m:02d}" for m in range(1, 13)], 'format': 'netcdf'}

    if 'oras5' in dataset:
        request.update({'product_type': ['consolidated'], 
                        'vertical_resolution': ['all_levels' if var_key == 'salinity' else 'single_level']})
        final_path = target_zip 
    else:
        request.update({'product_type': 'reanalysis', 'time': ['00:00', '06:00', '12:00', '18:00'], 
                        'area': AREA, 'day': [f"{d:02d}" for d in range(1, 32)]})
        if 'pressure-levels' in dataset: request.update({'pressure_level': ['850', '500', '200']})
        final_path = target_nc

    try:
        print(f"[*] Solicitando: {full_name}")
        c.retrieve(dataset, request).download(final_path)
        
        if final_path.endswith('.zip'):
            with zipfile.ZipFile(final_path, 'r') as z:
                ext_name = z.namelist()[0]
                z.extract(ext_name, CACHE_DIR)
                os.rename(os.path.join(CACHE_DIR, ext_name), target_nc)
            if os.path.exists(final_path): os.remove(final_path)
        return target_nc
    except Exception as e:
        print(f"❌ Erro em {full_name}: {e}"); return None

# 4. EXECUÇÃO DOS DOWNLOADS (FILAS SEPARADAS)
# ------------------------------------------------------------------------------
bench_start = time.time()
arquivos_era5 = []
arquivos_oras5 = []

print(f"🚀 Iniciando Filas de Download ENEB {ANO}...")

# Fila ERA5
for dset, dic, src in [('reanalysis-era5-single-levels', VARS_ERA5_SL, 'ERA5'),
                       ('reanalysis-era5-pressure-levels', VARS_ERA5_PL, 'ERA5')]:
    for k, v in dic.items():
        res = download_engine(dset, k, src, v)
        if res: arquivos_era5.append(res)

# Fila ORAS5
for k, v in VARS_ORAS5.items():
    res = download_engine('reanalysis-oras5', k, 'ORAS5', v)
    if res: arquivos_oras5.append(res)

# 5. UNIFICAÇÃO INDEPENDENTE (RESOLUÇÃO DE ERRO DE ÍNDICE)
# ------------------------------------------------------------------------------
def create_era5_zarr():
    if len(arquivos_era5) < 15: return
    print("⚙️ Processando ERA5 Zarr...")
    ds_list = []
    for f in arquivos_era5:
        ds = xr.open_dataset(f).drop_vars(['expver', 'number'], errors='ignore')
        if 'valid_time' in ds.coords: ds = ds.rename({'valid_time': 'time'})
        ds_d = ds.resample(time='1D').sum() if 'Precipitation' in f else ds.resample(time='1D').mean()
        ds_list.append(ds_d.compute())
    # Consolidated=True ajuda a evitar warnings de metadados
    xr.merge(ds_list, compat='override').to_zarr(os.path.join(DATA_DIR, f"ERA5_Zarr_{ANO}.zarr"), mode='w', consolidated=True)

def create_oras5_zarr():
    if len(arquivos_oras5) < 5: return
    print("⚙️ Processando ORAS5 Zarr (Grade Curvilínea)...")
    ds_list = []
    for f in arquivos_oras5:
        # Abrimos garantindo que não haja conflitos de coordenadas
        ds = xr.open_dataset(f)
        
        # Identifica a coordenada de tempo (ORAS5 varia entre time_counter e time)
        t_coord = 'time_counter' if 'time_counter' in ds.coords else 'time'
        
        # MUDANÇA: Selecionamos apenas as variáveis de dados para o resample
        # Isso evita que o scipy tente interpolar coordenadas lat/lon 2D (o que gera o erro de divide by zero)
        vars_to_keep = list(ds.data_vars)
        ds_resampled = ds[vars_to_keep].resample({t_coord: '1D'}).mean()
        
        if t_coord != 'time':
            ds_resampled = ds_resampled.rename({t_coord: 'time'})
        
        # Forçamos o carregamento para memória antes do append
        ds_list.append(ds_resampled.compute())
        ds.close()

    if ds_list:
        # MUDANÇA: Merge simples para evitar que o Xarray se perca em grades diferentes
        ds_final = xr.merge(ds_list)
        path_out = os.path.join(DATA_DIR, f"ORAS5_Zarr_{ANO}.zarr")
        ds_final.to_zarr(path_out, mode='w', consolidated=True)
        print(f"✅ ORAS5 Zarr finalizado em: {path_out}")

# FINALIZAÇÃO / CRIAÇÃO DE ARQUIVOS ZARR
create_era5_zarr()
create_oras5_zarr()

# 6. RELATÓRIO DE BENCHMARK TÉCNICO (REVISADO E DETALHADO)
# ------------------------------------------------------------------------------
def get_size_gb(path):
    if not os.path.exists(path): return 0
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024**3)
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024**3)

# Pesos do Cache (Bruto)
size_cache_era5 = sum(get_size_gb(f) for f in arquivos_era5)
size_cache_oras5 = sum(get_size_gb(f) for f in arquivos_oras5)
total_cache = size_cache_era5 + size_cache_oras5

# Pesos do Output (Transformado)
path_era5_zarr = os.path.join(DATA_DIR, f"ERA5_Zarr_{ANO}.zarr")
path_oras5_zarr = os.path.join(DATA_DIR, f"ORAS5_Zarr_{ANO}.zarr")
size_zarr_era5 = get_size_gb(path_era5_zarr)
size_zarr_oras5 = get_size_gb(path_oras5_zarr)
total_zarr = size_zarr_era5 + size_zarr_oras5

bench_end = time.time()
duracao = (bench_end - bench_start) / 60

print(f"\n{'='*80}")
print(f"📊 RELATÓRIO DE PERFORMANCE E ARMAZENAMENTO - ANO {ANO}")
print(f"{'='*80}")
print(f"⏱️  TEMPO TOTAL DE PROCESSAMENTO: {duracao:.2f} minutos")
print(f"{'-'*80}")
print(f"📦 PESO DOS ARQUIVOS EM CACHE (.NC):")
print(f"   ● ERA5 (15 variáveis):      {size_cache_era5:.4f} GB")
print(f"   ● ORAS5 (5 variáveis):      {size_cache_oras5:.4f} GB")
print(f"   ● TOTAL EM CACHE:           {total_cache:.4f} GB")
print(f"{'-'*80}")
print(f"🏗️  PESO DOS ARQUIVOS FINAIS (ZARR):")
print(f"   ● ERA5_Zarr_{ANO}:          {size_zarr_era5:.4f} GB")
print(f"   (Inclui levels: 850, 500, 200 + Superfície)")
print(f"   ● ORAS5_Zarr_{ANO}:         {size_zarr_oras5:.4f} GB")
print(f"   ● TOTAL EM ZARR:            {total_zarr:.4f} GB")
print(f"{'-'*80}")

if total_cache > 0:
    reducao = (1 - (total_zarr / total_cache)) * 100
    print(f"📉 EFICIÊNCIA DE COMPRESSÃO:   {reducao:.1f}% de economia de espaço")
else:
    print("📉 EFICIÊNCIA DE COMPRESSÃO:   N/A (Cache não detectado)")

print(f"{'='*80}")
print(f"🔮 ESTIMATIVA PARA SÉRIE HISTÓRICA (45 ANOS):")
print(f"   ● Espaço em Disco:          {(total_zarr * 45):.2f} GB")
print(f"{'='*80}\n")